In [15]:
import numpy as np
import os
import tensorflow.keras.models 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Dense, Flatten, Input, MaxPooling1D
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
from tensorflow.keras.optimizers import Adam

INPUT_FILE = 'hgdream/Sensl_FastOut_AveragePulse_1p8GHzBandwidth.feather'
SAVE_DIR = '1Conv_checkpoints_running'
LOG_FILE = '1Conv_3pulse_noise_tb23.csv'
MODEL_NAME_TEMPLATE = '1Conv_2pulse_noise.loss_{val_loss:01.5f}.e{epoch:03d}_deconv.h5'
SAVE_DIR2 = '1Conv_checkpoints_running2'
LOG_FILE2 = '1Conv_3pulse_noise_tb232.csv'
MODEL_NAME_TEMPLATE2 = '1Conv_2pulse_noise.loss_{val_loss:01.5f}.e{epoch:03d}_deconv2.h5'
SAVE_DIR3 = '1Conv_checkpoints_running3'
LOG_FILE3 = '1Conv_3pulse_noise_tb233.csv'
MODEL_NAME_TEMPLATE3 = '1Conv_2pulse_noise.loss_{val_loss:01.5f}.e{epoch:03d}_deconv3.h5'
NUM_EVENTS = 500
TIME_STEPS = 80 # divides width_cutoff
width_cutoff = 800
SCALE_FACTOR = 12.0 / 10.0
VAL_SPLIT = 0.1


X_data = np.load("X_Data_Bank.npy")
y_data = np.load("Y_Data_Bank.npy")

# --- Build Model ---
Train = True
if Train:
    model = Sequential([
        Input(shape=(TIME_STEPS, 1)),
        Conv1D(64, 3, activation='relu', padding='same'),
        Conv1D(32, 3, activation='relu', padding='same'),
        Conv1D(32, 3, activation='relu', padding='same'),
        MaxPooling1D(pool_size=3),
        Conv1D(32, 3, activation='relu', padding='same'),
        Conv1D(64, 3, activation='relu', padding='same'),
        Conv1D(1, 9, activation='relu', padding='same'),
        Flatten(),
        Dense(TIME_STEPS, activation='relu')
    ])

    model.summary()

    # --- Compile Model ---
    optimizer = Adam(amsgrad=True)
    model.compile(loss='mean_squared_logarithmic_error', optimizer=optimizer) #mean_squared_logarithmic_error

    # --- Setup Checkpoints & Callbacks ---
    if not os.path.isdir(SAVE_DIR3):
        os.makedirs(SAVE_DIR3)

    checkpoint_path = os.path.join(SAVE_DIR3, MODEL_NAME_TEMPLATE3)

    callbacks = [
        ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True, mode="min", verbose=1),
        EarlyStopping(monitor="val_loss", mode="min", patience=10),
        CSVLogger(LOG_FILE, append=True, separator=',')
    ]

    # --- Train Model ---
    model.fit(
        X_data, y_data,
        epochs=30,
        shuffle=True,
        validation_split=VAL_SPLIT,
        callbacks=callbacks
    )

    model.save('unpruned_model_pooling.h5')
else:
    from tensorflow.keras.models import load_model

    model = load_model('unpruned_model_pooling.h5')

Model: "sequential_13"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv1d_62 (Conv1D)          (None, 80, 64)            256       
                                                                 
 conv1d_63 (Conv1D)          (None, 80, 32)            6176      
                                                                 
 conv1d_64 (Conv1D)          (None, 80, 32)            3104      
                                                                 
 max_pooling1d_29 (MaxPoolin  (None, 26, 32)           0         
 g1D)                                                            
                                                                 
 conv1d_65 (Conv1D)          (None, 26, 32)            3104      
                                                                 
 conv1d_66 (Conv1D)          (None, 26, 64)            6208      
                                                     

Epoch 24/30
141/141 [==============================] - ETA: 0s - loss: 0.0185
Epoch 24: val_loss did not improve from 0.01907
141/141 [==============================] - 2s 16ms/step - loss: 0.0185 - val_loss: 0.0191
Epoch 25/30
139/141 [============================>.] - ETA: 0s - loss: 0.0185
Epoch 25: val_loss did not improve from 0.01907
141/141 [==============================] - 2s 15ms/step - loss: 0.0185 - val_loss: 0.0191
Epoch 26/30
137/141 [============================>.] - ETA: 0s - loss: 0.0183
Epoch 26: val_loss did not improve from 0.01907
141/141 [==============================] - 2s 15ms/step - loss: 0.0184 - val_loss: 0.0191
Epoch 27/30
136/141 [===========================>..] - ETA: 0s - loss: 0.0183
Epoch 27: val_loss did not improve from 0.01907
141/141 [==============================] - 2s 12ms/step - loss: 0.0183 - val_loss: 0.0192
Epoch 28/30
134/141 [===========================>..] - ETA: 0s - loss: 0.0183
Epoch 28: val_loss did not improve from 0.01907
141/141 [=